# Week 6b: Fine-Tuning — Hands-On Practice

**Applied Generative AI**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

---

## Learning Objectives

By the end of this notebook, you will be able to:

- **Apply LoRA fine-tuning** — Configure rank, alpha, target modules, and interpret trainable parameter counts
- **Use QLoRA** — 4-bit quantization + LoRA for memory-efficient fine-tuning
- **Track experiments with WandB** — Log metrics, compare runs, and visualize training curves
- **Evaluate fine-tuned models** — Accuracy, F1, confusion matrix, and base vs. fine-tuned comparison
- **Identify common failure modes** — Overfitting, catastrophic forgetting, data leakage

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q transformers datasets torch peft bitsandbytes wandb evaluate accelerate scikit-learn

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
import evaluate
import numpy as np
import torch
import torch.nn.functional as F

---
## 1. LoRA Fine-Tuning

We fine-tune **DistilBERT** on **SST-2** (Stanford Sentiment Treebank) using LoRA. SST-2 is a binary sentiment dataset of movie review sentences.

### 1.1 Load Dataset and Tokenizer

SST-2 comes from the GLUE benchmark. We use the `sentence` column for input and `label` for sentiment (0=negative, 1=positive).

In [ ]:
dataset = load_dataset("glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["sentence"], padding=True, truncation=True)

encoded_dataset = dataset.map(tokenize, batched=True)
encoded_dataset = encoded_dataset.rename_column("label", "labels")
encoded_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Train:", len(encoded_dataset["train"]), "| Validation:", len(encoded_dataset["validation"]))
sample = dataset["train"][0]["sentence"]
print("Sample:", sample[:80] + "..." if len(sample) > 80 else sample)

In [ ]:
# Use subset for faster runs (increase for better results)
train_subset = encoded_dataset["train"].shuffle(seed=42).select(range(1000))
eval_subset = encoded_dataset["validation"].select(range(200))

### 1.2 Load Base Model and Apply LoRA

**Target modules:** For DistilBERT, attention uses `q_lin` (query) and `v_lin` (value). We inject LoRA into these layers. You can also use `["query", "value"]` for models with different naming.

**r (rank):** Low-rank dimension. Higher = more capacity, more parameters. Start with 8.

**alpha:** Scaling factor. Often set to 2×r so effective scaling = alpha/r = 2.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

**Interpretation:** You should see ~1% trainable parameters. The rest are frozen. This is the key benefit of LoRA — we update only a small adapter while keeping the base model intact.

### 1.3 Train with LoRA

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./lora-distilbert",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"  # Set to "wandb" when using WandB
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
model.save_pretrained("lora-adapter-distilbert")
tokenizer.save_pretrained("lora-adapter-distilbert")
print("LoRA adapter saved.")

### 1.4 Load and Run Inference with LoRA Adapter

To use the fine-tuned model: load the base model, then merge the LoRA adapter. The adapter is small (~few MB) vs. full model (~250MB).

In [ ]:
tokenizer_inf = AutoTokenizer.from_pretrained("lora-adapter-distilbert")
base_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model_inf = PeftModel.from_pretrained(base_model, "lora-adapter-distilbert")
model_inf.eval()

In [ ]:
test_texts = ["This movie was fantastic!", "I hate that movie"]

for text in test_texts:
    inputs = tokenizer_inf(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model_inf(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    sentiment = "positive" if pred == 1 else "negative"
    print(f"Input: {text}\nPredicted: {sentiment} (prob: {probs[0][pred]:.3f})\n")

---
## 2. QLoRA: 4-bit Quantization + LoRA

QLoRA quantizes the base model to 4-bit, then applies LoRA in full precision. This reduces memory significantly. For larger models (7B+), this is essential on consumer GPUs.

**Note:** DistilBERT is small; QLoRA shines with 7B+ models. We demonstrate the setup here.

In [ ]:
# QLoRA config: 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model with quantization (requires CUDA for 4-bit)
if torch.cuda.is_available():
    model_qlora = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model_qlora = get_peft_model(model_qlora, lora_config)
    model_qlora.print_trainable_parameters()
else:
    print("QLoRA requires CUDA. Skipping on CPU.")

**Memory savings:** 4-bit vs 32-bit ≈ 8× smaller base weights. With LoRA adapters in fp16, total trainable memory is dominated by gradients and optimizer states for the small adapter.

---
## 3. Experiment Tracking with WandB

Weights & Biases (WandB) logs metrics, hyperparameters, and system info. Use `getpass` to securely input your API key.

In [ ]:
import getpass
import wandb

# Login — use getpass for security in Colab
wandb_key = getpass.getpass("WandB API key (or press Enter to skip): ")
if wandb_key:
    wandb.login(key=wandb_key)
    use_wandb = True
else:
    use_wandb = False
    print("WandB disabled. Set report_to='none' in TrainingArguments.")

In [ ]:
if use_wandb:
    wandb.init(project="fine-tuning-sst2", config={
        "model": "distilbert-base-uncased",
        "dataset": "sst2",
        "lora_r": 8,
        "lora_alpha": 16,
        "epochs": 10,
        "batch_size": 16,
    })

In [ ]:
# Example: Train with WandB logging
# Set report_to="wandb" in TrainingArguments and run trainer.train()
# Metrics (loss, eval_accuracy) will appear in WandB dashboard

args_wandb = TrainingArguments(
    output_dir="./lora-wandb",
    report_to="wandb" if use_wandb else "none",
    per_device_train_batch_size=16,
    num_train_epochs=5,
    logging_steps=10,
    eval_strategy="epoch",
)

print("To log to WandB, use Trainer with args_wandb and report_to='wandb'")

---
## 4. Evaluating Fine-Tuned Models

We compare **base model** (no fine-tuning) vs **LoRA fine-tuned** model on the validation set.

In [ ]:
def evaluate_model(model, dataset, device):
    model.eval()
    all_preds, all_labels = [], []
    for i in range(min(500, len(dataset))):
        row = dataset[i]
        inputs = {k: v.unsqueeze(0).to(device) for k, v in row.items() if k in ["input_ids", "attention_mask"]]}
        with torch.no_grad():
            out = model(**inputs)
        pred = out.logits.argmax(dim=-1).item()
        all_preds.append(pred)
        all_labels.append(row["labels"].item())
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return acc, all_preds, all_labels

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Base model (no fine-tuning)
base_model_eval = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
base_model_eval.to(device)
acc_base, preds_base, labels_base = evaluate_model(base_model_eval, eval_subset, device)

# Fine-tuned LoRA model
model_inf.to(device)
acc_ft, preds_ft, labels_ft = evaluate_model(model_inf, eval_subset, device)

print(f"Base model accuracy:  {acc_base:.4f}")
print(f"Fine-tuned accuracy:  {acc_ft:.4f}")
print(f"Improvement:          +{acc_ft - acc_base:.4f}")

In [ ]:
# F1 and confusion matrix
from sklearn.metrics import f1_score, confusion_matrix

f1_base = f1_score(labels_base, preds_base, average="binary")
f1_ft = f1_score(labels_ft, preds_ft, average="binary")

print("F1 scores:")
print(f"  Base:     {f1_base:.4f}")
print(f"  Fine-tuned: {f1_ft:.4f}")

print("\nConfusion matrix (fine-tuned):")
print(confusion_matrix(labels_ft, preds_ft))

---
## 5. Common Failure Modes

| Failure | Symptom | Mitigation |
|---------|---------|------------|
| **Overfitting** | Train loss ↓, eval loss ↑ | Fewer epochs, more data, dropout, weight decay |
| **Catastrophic forgetting** | Model forgets pre-trained knowledge | Lower LR, LoRA (freeze base), gradual unfreezing |
| **Data leakage** | Test data in train | Strict train/val/test splits, deduplication |
| **Label imbalance** | Poor minority class | Oversampling, class weights, balanced batches |

---
## 6. Exercises

### Exercise 1: LoRA Rank Comparison

Fine-tune with `r=4` and `r=16` (keep alpha=2*r). Compare trainable parameter count and validation accuracy. Does higher rank always help?

In [ ]:
# LoRA rank comparison — try r=4 and r=16
for r in [4, 16]:
    config = LoraConfig(r=r, lora_alpha=2*r, target_modules=["q_lin", "v_lin"], task_type=TaskType.SEQ_CLS)
    m = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
    m = get_peft_model(m, config)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"r={r}: {trainable:,} trainable parameters")

### Exercise 2: Build an Evaluation Suite

Create a small test set of 10–20 hand-written sentences (mix of positive/negative). Run your fine-tuned model and report accuracy. Add edge cases (sarcasm, neutral) and document failures.

In [ ]:
# Custom evaluation suite
custom_sentences = [
    ("Great film!", 1),
    ("Terrible acting.", 0),
    ("Not bad, could be better.", 0),  # edge case
]

model_inf.eval()
for text, true_label in custom_sentences:
    inputs = tokenizer_inf(text, return_tensors="pt").to(device)
    with torch.no_grad():
        pred = model_inf(**inputs).logits.argmax(dim=-1).item()
    correct = "✓" if pred == true_label else "✗"
    print(f"{correct} '{text}' -> pred={pred} (true={true_label})")

---
## 7. Responsible AI Reflection

**Training data bias propagation:**

Fine-tuned models inherit and can amplify biases in training data. If SST-2 or your custom data overrepresents certain demographics, genres, or sentiment patterns, the model will reflect that.

- **Action:** Audit your training data for representation and fairness
- **Action:** Evaluate on held-out data from different domains or demographics
- **Action:** Document known limitations in model cards

---
## 8. Summary & Next Steps

**You practiced:**
- LoRA fine-tuning with DistilBERT on SST-2
- QLoRA setup for memory-efficient training
- WandB for experiment tracking
- Evaluation (accuracy, F1, confusion matrix) and base vs. fine-tuned comparison
- Common failure modes and mitigation strategies

**Next:** Apply these techniques to your own domain — instruction tuning, domain adaptation, or task-specific classifiers.